# 🔬 Chronos-1 vs Chronos-2 Evaluation for RUL Prediction

**Objective**: Compare Chronos foundation models (v1 vs v2) for Remaining Useful Life prediction on C-MAPSS FD001

**Structure**:
- **Part A**: Data Preparation & Feature Selection (Correlation + AFICv)
- **Part B**: Chronos-1 Evaluation (embeddings + regression head)
- **Part C**: Chronos-2 Evaluation (native multivariate)
- **Part D**: Comparison Table

**Key Novelty**: Chronos-2 supports native multivariate forecasting, while Chronos-1 is univariate only.

**Author**: Fatima Khadija Benzine — March 2026

---
## 0. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q 'chronos-forecasting>=2.2' xgboost torch

import os
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import json, time, warnings
warnings.filterwarnings('ignore')

import torch
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M')
SAVE_DIR = f'/content/drive/MyDrive/PhD_results/Chronos_Eval_{TIMESTAMP}'
os.makedirs(SAVE_DIR, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
print(f"Save directory: {SAVE_DIR}")
print("Setup complete ✓")

---
## Part A: Data Preparation & Feature Selection

In [ ]:
# === A1: Load C-MAPSS FD001 ===

def load_cmapss_fd001():
    """Load C-MAPSS FD001 dataset from NASA repository"""
    base_url = "https://raw.githubusercontent.com/makinarocks/awesome-industrial-machine-datasets/master/data-explanation/CMAPSSData"
    
    col_names = ['unit', 'cycle'] + \
                [f'setting_{i}' for i in range(1, 4)] + \
                [f'sensor_{i}' for i in range(1, 22)]
    
    # Load train
    train_df = pd.read_csv(f"{base_url}/train_FD001.txt", sep=r'\s+', header=None, names=col_names)
    
    # Load test
    test_df = pd.read_csv(f"{base_url}/test_FD001.txt", sep=r'\s+', header=None, names=col_names)
    
    # Load RUL
    rul_df = pd.read_csv(f"{base_url}/RUL_FD001.txt", sep=r'\s+', header=None, names=['rul'])
    
    # Compute RUL for training data
    max_cycles = train_df.groupby('unit')['cycle'].max()
    train_df = train_df.merge(
        max_cycles.rename('max_cycle').reset_index(), on='unit'
    )
    train_df['rul'] = train_df['max_cycle'] - train_df['cycle']
    train_df.drop('max_cycle', axis=1, inplace=True)
    
    # Compute RUL for test data
    test_max = test_df.groupby('unit')['cycle'].max().reset_index()
    test_max['base_rul'] = rul_df['rul'].values
    test_df = test_df.merge(test_max, on='unit')
    test_df['rul'] = test_df['base_rul'] + (test_df['cycle'].max() - test_df['cycle'])
    
    # Recalculate properly
    test_df = test_df.drop(['cycle_max', 'base_rul'], axis=1, errors='ignore')
    test_df = pd.read_csv(f"{base_url}/test_FD001.txt", sep=r'\s+', header=None, names=col_names)
    
    rul_values = rul_df['rul'].values
    test_rul = []
    for unit in test_df['unit'].unique():
        unit_data = test_df[test_df['unit'] == unit]
        max_cycle = unit_data['cycle'].max()
        base_rul = rul_values[unit - 1]
        for cycle in unit_data['cycle']:
            test_rul.append(base_rul + (max_cycle - cycle))
    test_df['rul'] = test_rul
    
    return train_df, test_df

print("Loading C-MAPSS FD001...")
train_raw, test_raw = load_cmapss_fd001()
print(f"✓ Train: {len(train_raw)} samples, {train_raw['unit'].nunique()} units")
print(f"✓ Test: {len(test_raw)} samples, {test_raw['unit'].nunique()} units")

In [ ]:
# === A2: Preprocessing ===
RUL_CAP = 125  # Piecewise linear RUL cap
W = 30  # Window size

# Apply RUL cap
train_raw['rul'] = train_raw['rul'].clip(upper=RUL_CAP)
test_raw['rul'] = test_raw['rul'].clip(upper=RUL_CAP)

sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
setting_cols = [f'setting_{i}' for i in range(1, 4)]
feature_cols = sensor_cols + setting_cols

# Min-Max normalization
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
train_norm = train_raw.copy()
test_norm = test_raw.copy()

train_norm[feature_cols] = scaler.fit_transform(train_raw[feature_cols])
test_norm[feature_cols] = scaler.transform(test_raw[feature_cols])

print(f"✓ Preprocessing complete")
print(f"  RUL cap: {RUL_CAP}")
print(f"  Features: {len(feature_cols)} (21 sensors + 3 settings)")

In [ ]:
# === A3: Feature Selection ===

def variance_filter(df, feature_cols, threshold=0.01):
    """Remove low-variance features"""
    variances = df[feature_cols].var()
    selected = variances[variances > threshold].index.tolist()
    removed = [f for f in feature_cols if f not in selected]
    return selected, removed

def correlation_filter(df, feature_cols, threshold=0.95):
    """Remove highly correlated features"""
    corr_matrix = df[feature_cols].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    selected = [f for f in feature_cols if f not in to_drop]
    return selected, to_drop

def aficv_selection(df, feature_cols, target_col='rul', threshold=0.90):
    """AFICv: Accumulated Feature Importance with Cross-Validation"""
    from sklearn.model_selection import cross_val_score
    
    X = df[feature_cols].values
    y = df[target_col].values
    
    # Train XGBoost to get feature importances
    model = XGBRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
    model.fit(X, y)
    
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    
    # Select features covering threshold of total importance
    cumsum = np.cumsum(importances[indices])
    n_features = np.searchsorted(cumsum, threshold) + 1
    
    selected_indices = indices[:n_features]
    selected = [feature_cols[i] for i in selected_indices]
    
    return selected, cumsum[n_features-1]

# Strategy 1: Variance + Correlation filtering
print("=== Strategy 1: Variance + Correlation Filtering ===")
feats_var, removed_var = variance_filter(train_norm, feature_cols, threshold=0.001)
print(f"  Variance filter: {len(feature_cols)} → {len(feats_var)} (removed: {removed_var})")

feats_corr, removed_corr = correlation_filter(train_norm, feats_var, threshold=0.95)
print(f"  Correlation filter: {len(feats_var)} → {len(feats_corr)} (removed: {removed_corr})")

# Strategy 2: AFICv
print("\n=== Strategy 2: AFICv (90% importance coverage) ===")
feats_aficv, coverage = aficv_selection(train_norm, feature_cols, threshold=0.90)
print(f"  Selected: {len(feats_aficv)} features (coverage: {coverage:.1%})")
print(f"  Features: {feats_aficv}")

# Store feature sets
feature_sets = {
    'correlation': feats_corr,
    'aficv': feats_aficv
}

print("\n=== Summary ===")
for name, feats in feature_sets.items():
    print(f"  {name}: {len(feats)} features")

In [ ]:
# === A4: Create Sliding Windows ===

def create_sliding_windows(df, feature_cols, window_size=30):
    """Create sliding windows for each unit"""
    X_list, y_list, unit_list = [], [], []
    
    for unit in sorted(df['unit'].unique()):
        unit_data = df[df['unit'] == unit].sort_values('cycle')
        n_samples = len(unit_data)
        
        if n_samples < window_size:
            continue
        
        values = unit_data[feature_cols].values
        ruls = unit_data['rul'].values
        
        for start in range(n_samples - window_size + 1):
            end = start + window_size
            X_list.append(values[start:end])
            y_list.append(ruls[end - 1])  # RUL at last timestep
            unit_list.append(unit)
    
    return np.array(X_list), np.array(y_list), np.array(unit_list)

# Create windows for each feature set
data_dict = {}

for fs_name, features in feature_sets.items():
    X_train, y_train, units_train = create_sliding_windows(train_norm, features, W)
    X_test, y_test, units_test = create_sliding_windows(test_norm, features, W)
    
    data_dict[fs_name] = {
        'X_train': X_train, 'y_train': y_train, 'units_train': units_train,
        'X_test': X_test, 'y_test': y_test, 'units_test': units_test,
        'features': features
    }
    
    print(f"{fs_name}: X_train={X_train.shape}, X_test={X_test.shape}")

print("\n✓ Sliding windows created")

In [ ]:
# === A5: Evaluation Helpers ===

def nasa_score(y_true, y_pred):
    """NASA asymmetric scoring function (Saxena et al. 2008)"""
    d = y_pred - y_true
    score = np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1)
    return np.sum(score)

def evaluate_per_unit(y_true, y_pred, unit_labels):
    """Evaluate using last prediction per unit (standard C-MAPSS protocol)"""
    preds_last, true_last = [], []
    for u in sorted(set(unit_labels)):
        mask = unit_labels == u
        if mask.sum() > 0:
            preds_last.append(y_pred[mask][-1])
            true_last.append(y_true[mask][-1])
    
    preds_last = np.array(preds_last)
    true_last = np.array(true_last)
    
    rmse = np.sqrt(mean_squared_error(true_last, preds_last))
    score = nasa_score(true_last, preds_last)
    return rmse, score

# Store all results
ALL_RESULTS = []
print("Evaluation helpers ✓")

---
## Part B: Chronos-1 Evaluation

**Approach**: Extract embeddings using Chronos encoder → Train regression head for RUL prediction

**Note**: Chronos-1 is univariate, so we process each sensor separately and concatenate embeddings.

In [ ]:
# === B1: Load Chronos-1 Pipeline ===
from chronos import ChronosPipeline

# Available: chronos-t5-tiny (8M), mini (20M), small (46M), base (200M), large (710M)
CHRONOS1_MODEL = "amazon/chronos-t5-small"

print(f"Loading Chronos-1: {CHRONOS1_MODEL}...")
chronos1_pipeline = ChronosPipeline.from_pretrained(
    CHRONOS1_MODEL,
    device_map=DEVICE,
    torch_dtype=torch.float32
)
print("Chronos-1 loaded ✓")
print(f"  Architecture: T5 encoder-decoder")
print(f"  Capability: Univariate only")

In [ ]:
# === B2: Extract Chronos-1 Embeddings ===

def extract_chronos1_embeddings(pipeline, X, batch_size=32):
    """
    Extract Chronos-1 encoder embeddings.
    X shape: (n_samples, window_size, n_features)
    Since Chronos-1 is univariate, we process each feature and concatenate.
    """
    n_samples, window_size, n_features = X.shape
    all_embeddings = []
    
    for i in range(0, n_samples, batch_size):
        batch = X[i:i+batch_size]  # (batch, window, features)
        batch_embeddings = []
        
        for feat_idx in range(n_features):
            # Get this feature for all samples in batch
            feat_data = batch[:, :, feat_idx]  # (batch, window)
            ts_tensor = torch.tensor(feat_data, dtype=torch.float32).to(DEVICE)
            
            with torch.no_grad():
                emb = pipeline.embed(ts_tensor)  # (batch, seq_len, embed_dim)
                # Mean pool over sequence length
                emb_pooled = emb.mean(dim=1).cpu().numpy()  # (batch, embed_dim)
            
            batch_embeddings.append(emb_pooled)
        
        # Concatenate all feature embeddings: (batch, n_features * embed_dim)
        combined = np.concatenate(batch_embeddings, axis=1)
        all_embeddings.append(combined)
        
        if (i // batch_size) % 10 == 0:
            print(f"    Processed {min(i+batch_size, n_samples)}/{n_samples} samples")
    
    return np.vstack(all_embeddings)

print("Chronos-1 embedding function defined ✓")

In [ ]:
# === B3: Chronos-1 Evaluation ===

chronos1_results = {}

for fs_name in feature_sets.keys():
    print(f"\n{'='*60}")
    print(f"Chronos-1 + {fs_name.upper()} ({len(feature_sets[fs_name])} features)")
    print('='*60)
    
    data = data_dict[fs_name]
    
    # Extract embeddings
    print("Extracting train embeddings...")
    start_time = time.time()
    X_train_emb = extract_chronos1_embeddings(chronos1_pipeline, data['X_train'])
    train_time = time.time() - start_time
    print(f"  Shape: {X_train_emb.shape}, Time: {train_time:.1f}s")
    
    print("Extracting test embeddings...")
    start_time = time.time()
    X_test_emb = extract_chronos1_embeddings(chronos1_pipeline, data['X_test'])
    test_time = time.time() - start_time
    print(f"  Shape: {X_test_emb.shape}, Time: {test_time:.1f}s")
    
    # Train regression heads
    heads = [
        ('Ridge', Ridge(alpha=1.0)),
        ('MLP', MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=500, random_state=42)),
        ('XGBoost', XGBRegressor(n_estimators=100, max_depth=5, random_state=42))
    ]
    
    for head_name, head_model in heads:
        print(f"\n  Training {head_name} head...")
        head_model.fit(X_train_emb, data['y_train'])
        
        y_pred = head_model.predict(X_test_emb)
        rmse, score = evaluate_per_unit(data['y_test'], y_pred, data['units_test'])
        
        result = {
            'Model': 'Chronos-1',
            'Feature_Set': fs_name,
            'Head': head_name,
            'Test_RMSE': round(rmse, 2),
            'Test_Score': round(score, 1),
            'Time_sec': round(train_time + test_time, 1)
        }
        ALL_RESULTS.append(result)
        chronos1_results[f"{fs_name}_{head_name}"] = result
        
        print(f"    RMSE: {rmse:.2f}, Score: {score:.1f}")

print("\n✓ Chronos-1 evaluation complete")

---
## Part C: Chronos-2 Evaluation

**Advantage**: Chronos-2 supports native multivariate forecasting via group attention mechanism.

In [ ]:
# === C1: Load Chronos-2 Pipeline ===
from chronos import Chronos2Pipeline

CHRONOS2_MODEL = "amazon/chronos-2"

print(f"Loading Chronos-2: {CHRONOS2_MODEL}...")
chronos2_pipeline = Chronos2Pipeline.from_pretrained(
    CHRONOS2_MODEL,
    device_map=DEVICE
)
print("Chronos-2 loaded ✓")
print(f"  Parameters: 120M")
print(f"  Architecture: Encoder-only + Group Attention")
print(f"  Capabilities: Univariate, Multivariate, Covariates")

In [ ]:
# === C2: Extract Chronos-2 Embeddings (Multivariate) ===

def extract_chronos2_embeddings(pipeline, X, batch_size=32):
    """
    Extract Chronos-2 encoder embeddings.
    Chronos-2 can process multivariate series natively.
    X shape: (n_samples, window_size, n_features)
    """
    n_samples, window_size, n_features = X.shape
    all_embeddings = []
    
    for i in range(0, n_samples, batch_size):
        batch = X[i:i+batch_size]  # (batch, window, features)
        
        # Transpose to (batch, features, window) for Chronos-2
        batch_t = np.transpose(batch, (0, 2, 1))  # (batch, features, window)
        
        # Process each sample (Chronos-2 expects (n_series, seq_len))
        batch_embeddings = []
        for sample in batch_t:
            ts_tensor = torch.tensor(sample, dtype=torch.float32).to(DEVICE)
            
            with torch.no_grad():
                emb = pipeline.embed(ts_tensor)  # (n_features, n_patches, embed_dim)
                # Mean pool over features and patches
                emb_pooled = emb.mean(dim=(0, 1)).cpu().numpy()  # (embed_dim,)
            
            batch_embeddings.append(emb_pooled)
        
        all_embeddings.extend(batch_embeddings)
        
        if (i // batch_size) % 10 == 0:
            print(f"    Processed {min(i+batch_size, n_samples)}/{n_samples} samples")
    
    return np.array(all_embeddings)

print("Chronos-2 embedding function defined ✓")

In [ ]:
# === C3: Chronos-2 Evaluation ===

chronos2_results = {}

for fs_name in feature_sets.keys():
    print(f"\n{'='*60}")
    print(f"Chronos-2 + {fs_name.upper()} ({len(feature_sets[fs_name])} features)")
    print('='*60)
    
    data = data_dict[fs_name]
    
    # Extract embeddings (multivariate)
    print("Extracting train embeddings (multivariate)...")
    start_time = time.time()
    X_train_emb = extract_chronos2_embeddings(chronos2_pipeline, data['X_train'])
    train_time = time.time() - start_time
    print(f"  Shape: {X_train_emb.shape}, Time: {train_time:.1f}s")
    
    print("Extracting test embeddings...")
    start_time = time.time()
    X_test_emb = extract_chronos2_embeddings(chronos2_pipeline, data['X_test'])
    test_time = time.time() - start_time
    print(f"  Shape: {X_test_emb.shape}, Time: {test_time:.1f}s")
    
    # Train regression heads
    heads = [
        ('Ridge', Ridge(alpha=1.0)),
        ('MLP', MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=500, random_state=42)),
        ('XGBoost', XGBRegressor(n_estimators=100, max_depth=5, random_state=42))
    ]
    
    for head_name, head_model in heads:
        print(f"\n  Training {head_name} head...")
        head_model.fit(X_train_emb, data['y_train'])
        
        y_pred = head_model.predict(X_test_emb)
        rmse, score = evaluate_per_unit(data['y_test'], y_pred, data['units_test'])
        
        result = {
            'Model': 'Chronos-2',
            'Feature_Set': fs_name,
            'Head': head_name,
            'Test_RMSE': round(rmse, 2),
            'Test_Score': round(score, 1),
            'Time_sec': round(train_time + test_time, 1)
        }
        ALL_RESULTS.append(result)
        chronos2_results[f"{fs_name}_{head_name}"] = result
        
        print(f"    RMSE: {rmse:.2f}, Score: {score:.1f}")

print("\n✓ Chronos-2 evaluation complete")

---
## Part D: Comparison

In [ ]:
# === D1: Results Table ===
results_df = pd.DataFrame(ALL_RESULTS)
results_df = results_df.sort_values('Test_RMSE')

print("\n" + "="*80)
print("RESULTS: Chronos-1 vs Chronos-2 on C-MAPSS FD001")
print("="*80)
print(results_df.to_string(index=False))

# Add baselines from IMSA2025 paper
print("\n" + "="*80)
print("BASELINES (from IMSA2025 paper)")
print("="*80)
baselines = [
    {'Model': 'XGBoost', 'RMSE': 14.64, 'Score': 307.13},
    {'Model': 'Classical LSTM', 'RMSE': 16.14, 'Score': 397.75},
    {'Model': 'Proposed LSTM', 'RMSE': 14.32, 'Score': 325.05},
    {'Model': 'Chronos + XGBoost', 'RMSE': 15.93, 'Score': 379.02},
    {'Model': 'Chronos + ANN', 'RMSE': 14.92, 'Score': 641.58},
    {'Model': 'Finetuned Chronos', 'RMSE': 65.11, 'Score': 79257.80},
]
baselines_df = pd.DataFrame(baselines)
print(baselines_df.to_string(index=False))

In [ ]:
# === D2: Visualization ===
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Prepare data for plotting
chronos_df = results_df.copy()
chronos_df['Label'] = chronos_df['Model'] + '\n' + chronos_df['Feature_Set'] + '\n' + chronos_df['Head']

colors = ['#FF6B6B' if 'Chronos-1' in m else '#4ECDC4' for m in chronos_df['Model']]

# RMSE
ax1 = axes[0]
bars1 = ax1.barh(chronos_df['Label'], chronos_df['Test_RMSE'], color=colors, alpha=0.8)
ax1.axvline(x=14.32, color='green', linestyle='--', linewidth=2, label='IMSA2025 LSTM: 14.32')
ax1.axvline(x=14.64, color='orange', linestyle='--', linewidth=2, label='IMSA2025 XGBoost: 14.64')
ax1.set_xlabel('Test RMSE (lower is better)', fontsize=12)
ax1.set_title('RMSE Comparison', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right')

for bar, val in zip(bars1, chronos_df['Test_RMSE']):
    ax1.text(val + 0.1, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center')

# NASA Score
ax2 = axes[1]
bars2 = ax2.barh(chronos_df['Label'], chronos_df['Test_Score'], color=colors, alpha=0.8)
ax2.axvline(x=325.05, color='green', linestyle='--', linewidth=2, label='IMSA2025 LSTM: 325')
ax2.axvline(x=307.13, color='orange', linestyle='--', linewidth=2, label='IMSA2025 XGBoost: 307')
ax2.set_xlabel('NASA Score (lower is better)', fontsize=12)
ax2.set_title('NASA Score Comparison', fontsize=14, fontweight='bold')
ax2.legend(loc='lower right')

for bar, val in zip(bars2, chronos_df['Test_Score']):
    ax2.text(val + 5, bar.get_y() + bar.get_height()/2, f'{val:.0f}', va='center')

# Legend for colors
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#FF6B6B', label='Chronos-1'),
                   Patch(facecolor='#4ECDC4', label='Chronos-2')]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, fontsize=12)

plt.suptitle('Chronos-1 vs Chronos-2 for RUL Prediction (C-MAPSS FD001)', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/chronos_comparison_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === D3: Save Results ===
results_df.to_csv(f'{SAVE_DIR}/chronos_results_{TIMESTAMP}.csv', index=False)

# Best result
best = results_df.iloc[0].to_dict()
with open(f'{SAVE_DIR}/best_result_{TIMESTAMP}.json', 'w') as f:
    json.dump(best, f, indent=2, default=str)

# Summary
summary = {
    'dataset': 'C-MAPSS FD001',
    'window_size': W,
    'rul_cap': RUL_CAP,
    'chronos1_model': CHRONOS1_MODEL,
    'chronos2_model': CHRONOS2_MODEL,
    'best_model': best['Model'],
    'best_features': best['Feature_Set'],
    'best_head': best['Head'],
    'best_rmse': best['Test_RMSE'],
    'best_score': best['Test_Score']
}
with open(f'{SAVE_DIR}/summary_{TIMESTAMP}.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Results saved to {SAVE_DIR}")
print(f"\n{'='*60}")
print("BEST RESULT:")
print(f"  Model: {best['Model']}")
print(f"  Features: {best['Feature_Set']}")
print(f"  Head: {best['Head']}")
print(f"  RMSE: {best['Test_RMSE']}")
print(f"  Score: {best['Test_Score']}")
print('='*60)

---
## Summary

### Experimental Setup
- **Dataset**: C-MAPSS FD001 (100 train units, 100 test units)
- **Window size**: 30 cycles
- **Feature selection**: Variance+Correlation, AFICv (90% importance)
- **Metrics**: RMSE and NASA Score (per-unit last prediction)

### Model Comparison

| Aspect | Chronos-1 | Chronos-2 |
|--------|-----------|----------|
| Architecture | T5 Encoder-Decoder | Encoder + Group Attention |
| Multivariate | ❌ (process separately) | ✅ Native |
| Parameters | 8M-710M (configurable) | 120M |
| Context Length | 512 | 8192 |

### Key Findings
1. Chronos-2's native multivariate support may provide better representations
2. Embedding + regression head approach works for both versions
3. Compare with IMSA2025 baselines to assess performance

### Files Saved
```
PhD_results/Chronos_Eval_YYYYMMDD_HHMM/
    chronos_results_*.csv
    chronos_comparison_*.png
    best_result_*.json
    summary_*.json
```